# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s0m-a/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### The Contract (5 plain-words answers)
***What one row means***: One row = one content page, on one specific day. So if a page exists for 30 days, it has 30 rows —one per day.

***Which table(s) will be used***:
 - fact_content_daily_performance :  the main table, broken up by month
 - dim_content : extra info about each page (like word count, age, content type), joined in when needed
 - fact_content_query_90d : data on how many different search queries send traffic to each page

***Which time window?***: 
One specific month: March 2026. Inside that month:

- I build my features (the inputs) from days 31–60 before the month ends
- I build my label (the thing I'm trying to predict) from the last 30 days of the month
So the features come from an earlier chunk of time than the label — that's on purpose, so I'm not accidentally using future data to predict the past.

***What you'd predict (label/proxy):*** : Whether a page's traffic dropped by more than 20% compared to the month before.
This is a binary proxy label built from observable GSC counts, not from trend_direction or trend_pct (which are the starter CSV's label sources and must stay excluded from the warehouse work too).

***One thing you deliberately exclude:***:
Rows where GA4 data wasn't actually collected — those rows just get filled in with zero instead of real data. I'm filtering those out, because a fake zero isn't the same as "genuinely no traffic," and mixing them in would quietly bias the results


In [45]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field                                    | Bucket   | Notes                                             |
|-------------------------------------------|----------|----------------------------------------------------|
| gsc_impressions (prev_30 window)           | Feature  | Knowable before the label period                    |
| gsc_avg_position (prev_30 window)          | Feature  | Position already set before we predict              |
| gsc_clicks (prev_30 window)                | Feature  | Historical, pre-decision                            |
| visible_queries / rare_share / anon_share  | Feature  | From query table _prev30 cols                       |
| imp_last30 < 0.8 × imp_prev30              | Label    | The thing we predict                                |
| client_hash_id, content_hash_id            | Context  | Join/split only, never a model input                |
| gsc_impressions (last_30 window)           | Excluded | Future information — it IS the label                |
| ga4_data_available IS NOT TRUE rows        | Excluded | Zero-filled artefact, not real engagement            |
| trend_direction, trend_pct                 | Excluded | Derived from the label signal — direct leakage       |

In [48]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [50]:
import os
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...):  ········


In [51]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
MONTH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{MONTH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Duplicate grain rows: {len(grain_check)}")  # expect 0
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows: 0


,report_date,client_hash_id,content_hash_id,c


In [52]:
# Query 2 — Row count and date span of this partition:
span = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           MIN(report_date) AS earliest,
           MAX(report_date) AS latest
    FROM read_parquet('{MONTH}')
""").df()
span


,total_rows,earliest,latest
0,9841378,2026-03-01,2026-03-31


In [53]:
#Query 3 — Availability filter with IS TRUE:
avail = con.sql(f"""
    SELECT COUNT(*) AS rows_with_ga4
    FROM read_parquet('{MONTH}')
    WHERE ga4_data_available IS TRUE
""").df()
print(f"Rows where GA4 is available: {avail['rows_with_ga4'][0]:,}")
avail


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where GA4 is available: 413,966


,rows_with_ga4
0,413966


***Five-Feature Frame***
| # | Feature              | How to build it                                                                 | Available when?                                                                                                    |
|---|-----------------------|----------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------|
| 1 | imp_prev30            | SUM(gsc_impressions) for report_date ≤ 2026-03-01 (days 31–60 back from month end) | Knowable at decision moment — the prev-30 window closes on 2026-03-01, before the label period (2026-03-02 → 2026-03-31) opens |
| 2 | pos_prev30            | AVG(gsc_avg_position) over that same prev-30 window                              | Knowable — GSC position is reported in arrears; the full window is settled before prediction                          |
| 3 | clk_prev30            | SUM(gsc_clicks) over the prev-30 window                                          | Knowable — same closed window as imp_prev30; no future information touched                                            |
| 4 | imp_ratio             | imp_last30 / NULLIF(imp_prev30, 0)                                               | **Label — DO NOT use as a feature.** Put this in the label cell only. (This is the trap column for the leakage experiment) |
| 5 | days_active_prev30    | COUNT(DISTINCT report_date) where gsc_impressions > 0 in the prev-30 window       | Knowable — counts only days already in the past; measures how consistently the page appeared in search before the label window |

In [55]:
# Build the five-feature frame — query Feb (prev-30) and Mar (last-30) together
# Features: February window (closed before prediction point)
# Label:    March window (the outcome we predict)

# Point to the entire directory (like in nb 03)
ALL_MONTHS = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date <  '2026-03-01' THEN gsc_impressions  ELSE 0 END) AS imp_prev30,
        AVG(CASE WHEN report_date <  '2026-03-01' THEN gsc_avg_position END)        AS pos_prev30,
        SUM(CASE WHEN report_date <  '2026-03-01' THEN gsc_clicks       ELSE 0 END) AS clk_prev30,
        COUNT(DISTINCT CASE WHEN report_date < '2026-03-01'
              AND gsc_impressions > 0 THEN report_date END)                          AS days_active_prev30,
        SUM(CASE WHEN report_date >= '2026-03-01' THEN gsc_impressions  ELSE 0 END) AS imp_last30
    FROM read_parquet('{ALL_MONTHS}')
    WHERE report_date >= '2026-02-01' AND report_date <= '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
    HAVING imp_prev30 > 0
""").df()

# Binary label: did impressions drop more than 20%?
frame['is_declining'] = (frame['imp_last30'] < 0.8 * frame['imp_prev30']).astype(int)

print(f"Feature frame: {len(frame):,} rows")
print(f"Decline rate:  {frame['is_declining'].mean():.1%}")
frame[['imp_prev30', 'pos_prev30', 'clk_prev30', 'days_active_prev30', 'imp_last30', 'is_declining']].head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 153,559 rows
Decline rate:  30.1%


,imp_prev30,pos_prev30,clk_prev30,days_active_prev30,imp_last30,is_declining
0,70.0,7.564827,0.0,22,43.0,1
1,817.0,1.480969,1.0,28,988.0,0
2,1235.0,4.949239,1.0,28,1177.0,0
3,216.0,5.633971,0.0,28,487.0,0
4,9.0,26.142857,1.0,7,32.0,0


## Trap / Leakage Experiment 


In [57]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd

model_data = frame.dropna(subset=['imp_prev30','pos_prev30','clk_prev30','days_active_prev30'])

# --- DELIBERATE LEAK: imp_last30 is the label numerator ---
leaked_cols = ['imp_prev30', 'pos_prev30', 'clk_prev30', 'days_active_prev30', 'imp_last30']
X_leak = model_data[leaked_cols]
y      = model_data['is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)
clf_leak = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
leaked_score = accuracy_score(y_te, clf_leak.predict(X_te))
print(f"Leaked accuracy (imp_last30 included): {leaked_score:.3f}")


Leaked accuracy (imp_last30 included): 0.994


In [58]:
# --- HONEST: remove the leaky column ---
honest_cols = ['imp_prev30', 'pos_prev30', 'clk_prev30', 'days_active_prev30']
X_honest = model_data[honest_cols]
X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
clf_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
honest_score = accuracy_score(y_te, clf_honest.predict(X_te))
print(f"Honest accuracy (no leak):             {honest_score:.3f}")
print(f"\nLeakage inflated accuracy by {leaked_score - honest_score:.3f} — the model was given the answer.")


Honest accuracy (no leak):             0.693

Leakage inflated accuracy by 0.301 — the model was given the answer.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: GA4 coverage is near-zero for this slice.

Of the 9,841,378 rows in month=2026-03, only 413,966 (4.2%) have ga4_data_available IS TRUE. Any feature built from GA4 columns (sessions, pageviews, engagement rate) is silently missing for the other 95.8% of rows. This forces the lane to rely entirely on GSC signals (impressions, clicks, position). The consequence: the model cannot distinguish between a page that had zero sessions because it earned no organic traffic and one that had zero sessions because the client had never connected GA4 — the data cannot tell them apart at this scale.

In [61]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.